# Topic 2: Schema — StructType, StructField & Data Types

> **Phase 3 → Week 3 → Topic 2**

---

## The Passport Analogy

A passport has a fixed **schema**:
- Field: `surname` → Type: text, max 39 chars, not null
- Field: `date_of_birth` → Type: date, not null
- Field: `nationality` → Type: text, not null
- Field: `middle_name` → Type: text, **nullable**

The schema tells you:
1. Which fields exist (column names)
2. What type of data each holds (data types)
3. Whether the field can be empty (nullable)

Spark `StructType` is exactly this — the passport definition for your DataFrame.

---

## What is a Schema?

A schema defines the **structure** of a DataFrame:
- Column names
- Data types for each column
- Whether each column can contain nulls

```
StructType([                              ← the schema (list of fields)
    StructField("id",     IntegerType(), nullable=False),   ← one column
    StructField("name",   StringType(),  nullable=True),
    StructField("salary", DoubleType(),  nullable=True),
])
```

---

## All Spark Data Types

### Numeric
| Type | Python equivalent | Notes |
|------|------------------|-------|
| `ByteType()` | int | -128 to 127 |
| `ShortType()` | int | -32768 to 32767 |
| `IntegerType()` | int | -2B to 2B |
| `LongType()` | int | -9.2×10¹⁸ to 9.2×10¹⁸ ← default for Python int |
| `FloatType()` | float | 32-bit float |
| `DoubleType()` | float | 64-bit float ← default for Python float |
| `DecimalType(p,s)` | Decimal | Exact decimal — use for money! |

### String & Binary
| Type | Notes |
|------|-------|
| `StringType()` | UTF-8 string |
| `BinaryType()` | byte array |

### Boolean
| Type | Notes |
|------|-------|
| `BooleanType()` | True / False |

### Date & Time
| Type | Notes |
|------|-------|
| `DateType()` | Date only (no time) |
| `TimestampType()` | Date + Time (microsecond precision) |
| `TimestampNTZType()` | Timestamp without timezone (Spark 3.4+) |

### Complex Types
| Type | Example | Notes |
|------|---------|-------|
| `ArrayType(elementType)` | `[1, 2, 3]` | List of elements |
| `MapType(keyType, valueType)` | `{"a": 1}` | Key-value pairs |
| `StructType([...])` | nested row | Nested struct |

---

## `inferSchema` vs Explicit Schema

### inferSchema=True — Convenient but Slow & Risky

```python
df = spark.read.option("inferSchema", True).csv("data.csv")
```

**What Spark does:** reads the entire file TWICE — first pass to guess types, second pass to load data.

**Problems:**
- Doubles read time on large files
- May guess wrong (e.g., `"001"` → String instead of Integer)
- `"2024-01-15"` → inferred as String, not DateType
- `null` values in a column → Spark may infer wrong type
- Schema can change between runs if data changes → silent bugs in pipelines!

### Explicit Schema — Best Practice for Production

```python
schema = StructType([...])
df = spark.read.schema(schema).csv("data.csv")
```

**Benefits:**
- Reads file ONCE → 2x faster on large files
- Schema is guaranteed — no surprises
- Type errors caught immediately (bad record handling)
- Pipeline is reproducible — same schema every run
- Self-documenting code

**Rule: Always use explicit schema in production pipelines.**

---

## Interview Cheat Sheet

**Q: What is StructType in Spark?**
> StructType is Spark's schema class — a collection of StructField objects that define the column names, data types, and nullable constraints of a DataFrame. It's the equivalent of a table definition in SQL.

**Q: Why should you avoid inferSchema in production?**
> inferSchema reads the file twice (doubling I/O), may guess types incorrectly (e.g., treating IDs as integers), and the inferred schema can silently change if the data changes. In production, use an explicit StructType schema to guarantee correctness, improve performance, and make the pipeline self-documenting.

**Q: What is the difference between `IntegerType` and `LongType`?**
> IntegerType is 32-bit (max ~2 billion). LongType is 64-bit (max ~9.2×10¹⁸). Python's default `int` maps to LongType in Spark. Use IntegerType explicitly when you know values fit in 32-bit to save memory (half the storage per value).

**Q: When should you use `DecimalType` instead of `DoubleType`?**
> For monetary values. `DoubleType` (floating point) has rounding errors: `0.1 + 0.2 ≠ 0.3` in binary floating point. `DecimalType(18, 2)` is exact — stores up to 18 total digits with 2 decimal places, like SQL's DECIMAL. Always use DecimalType for currency, financial calculations, or any value where precision matters.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week3 - Schema") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark ready")

## Part 1: Building Schemas — All Data Types

In [ ]:
# Full schema with all common types
full_schema = StructType([
    # Numeric
    StructField("emp_id",     IntegerType(),        nullable=False),
    StructField("emp_long_id",LongType(),            nullable=True),
    StructField("score",      FloatType(),           nullable=True),
    StructField("salary",     DoubleType(),          nullable=True),
    StructField("bonus",      DecimalType(10, 2),    nullable=True),   # exact decimal!

    # String & Boolean
    StructField("name",       StringType(),          nullable=False),
    StructField("is_active",  BooleanType(),         nullable=True),

    # Date & Time
    StructField("join_date",  DateType(),            nullable=True),
    StructField("last_login", TimestampType(),        nullable=True),

    # Complex types
    StructField("skills",     ArrayType(StringType()), nullable=True),
    StructField("metadata",   MapType(StringType(), StringType()), nullable=True),

    # Nested struct
    StructField("address",    StructType([
        StructField("city",    StringType(), nullable=True),
        StructField("country", StringType(), nullable=True),
        StructField("pincode", StringType(), nullable=True),
    ]),                                      nullable=True),
])

print("Full schema definition:")
for field in full_schema.fields:
    print(f"  {field.name:<15} {str(field.dataType):<30} nullable={field.nullable}")

In [ ]:
# Create DataFrame with complex types
from datetime import date, datetime
from decimal import Decimal

complex_data = [
    (
        1, 100001, 8.5, 95000.0, Decimal("9500.00"),
        "Alice", True,
        date(2021, 3, 15), datetime(2024, 1, 10, 9, 30, 0),
        ["Python", "Spark", "SQL"],
        {"team": "platform", "level": "senior"},
        ("Bangalore", "India", "560001")
    ),
    (
        2, 100002, 7.2, 88000.0, Decimal("8800.00"),
        "Bob", True,
        date(2020, 7, 1), datetime(2024, 1, 9, 14, 0, 0),
        ["Java", "Scala"],
        {"team": "data", "level": "mid"},
        ("New York", "USA", "10001")
    ),
]

cdf = spark.createDataFrame(complex_data, full_schema)
cdf.printSchema()
cdf.show(truncate=False)

In [ ]:
# Accessing complex type columns
print("Access ArrayType — skills column:")
cdf.select("name", "skills", F.col("skills")[0].alias("first_skill")).show()

print("Access MapType — metadata column:")
cdf.select("name", F.col("metadata")["team"].alias("team")).show()

print("Access Nested StructType — address column:")
cdf.select("name", "address.city", "address.country").show()

## Part 2: inferSchema vs Explicit Schema — The Speed Difference

In [ ]:
import time

csv_path = "data/employees.csv"

# inferSchema — reads file twice
t0 = time.time()
df_infer = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)
infer_time = time.time() - t0

print("=== inferSchema ===")
print(f"Time: {infer_time:.4f}s")
df_infer.printSchema()

# Problems with inferSchema:
print("Issues with inferred schema:")
print(f"  'id' column type: {df_infer.schema['id'].dataType}  ← should be IntegerType")
print(f"  'join_date' type: {df_infer.schema['join_date'].dataType}  ← should be DateType!")

In [ ]:
# Explicit schema — reads file once, correct types
explicit_schema = StructType([
    StructField("id",        IntegerType(), nullable=False),
    StructField("name",      StringType(),  nullable=True),
    StructField("dept",      StringType(),  nullable=True),
    StructField("salary",    IntegerType(), nullable=True),
    StructField("country",   StringType(),  nullable=True),
    StructField("join_date", DateType(),    nullable=True),
    StructField("active",    BooleanType(), nullable=True),
])

t0 = time.time()
df_explicit = spark.read \
    .schema(explicit_schema) \
    .option("header", True) \
    .option("dateFormat", "yyyy-MM-dd") \
    .csv(csv_path)
explicit_time = time.time() - t0

print("=== Explicit Schema ===")
print(f"Time: {explicit_time:.4f}s")
df_explicit.printSchema()

print("\njoin_date is now properly DateType:")
df_explicit.select("name", "join_date", F.year("join_date").alias("year")).show()

## Part 3: Schema Manipulation — Useful Operations

In [ ]:
df = df_explicit

# schema property — get the StructType object
schema_obj = df.schema
print("df.schema (StructType object):")
print(schema_obj)
print()

# Access individual field
print("Individual field info:")
field = df.schema["salary"]
print(f"  Name: {field.name}, Type: {field.dataType}, Nullable: {field.nullable}")

# Get schema as JSON string (useful for saving/loading schemas)
import json
schema_json = df.schema.json()
print(f"\nSchema as JSON (first 200 chars):")
print(schema_json[:200] + "...")

# Rebuild schema from JSON (useful in pipelines)
rebuilt = StructType.fromJson(json.loads(schema_json))
print(f"\nRebuilt from JSON matches: {rebuilt == df.schema}")

In [ ]:
# Casting types at read time or with withColumn
print("Casting types on an existing column:")

# Cast salary from Int to Double
df_cast = df.withColumn("salary_dbl", F.col("salary").cast(DoubleType()))
df_cast.select("name", "salary", "salary_dbl").show(3)

# Cast using string type name (simpler syntax)
df_cast2 = df.withColumn("salary_str", F.col("salary").cast("string"))
df_cast2.select("name", "salary", "salary_str").show(3)

## Part 4: Handling Nulls and Bad Records

In [ ]:
# nullable field demonstration
nullable_data = [
    (1, "Alice",  95000),
    (2, None,     88000),   # null name
    (3, "Carol",  None),    # null salary
    (4, None,     None),    # both null
]

null_schema = StructType([
    StructField("id",     IntegerType(), nullable=False),
    StructField("name",   StringType(),  nullable=True),   # can be null
    StructField("salary", IntegerType(), nullable=True),   # can be null
])

df_nulls = spark.createDataFrame(nullable_data, null_schema)
print("DataFrame with nulls:")
df_nulls.show()

# null checks
print("Rows where name IS NULL:")
df_nulls.filter(F.col("name").isNull()).show()

print("Rows where salary IS NOT NULL:")
df_nulls.filter(F.col("salary").isNotNull()).show()

# dropna
print("dropna() — drop rows with ANY null:")
df_nulls.dropna().show()

print("dropna(subset=['name']) — drop only where name is null:")
df_nulls.dropna(subset=["name"]).show()

# fillna
print("fillna — replace nulls with defaults:")
df_nulls.fillna({"name": "Unknown", "salary": 0}).show()

## Exercises

1. Create a schema for an e-commerce order: `order_id (int), customer_name (string), product_name (string), quantity (int), unit_price (decimal 10,2), order_date (date), is_returned (boolean)`. Create a DataFrame with 3 sample rows using this schema.
2. Read `data/employees.csv` without `inferSchema`. Compare the `join_date` column type with vs without an explicit `DateType` schema.
3. Create a DataFrame with an `ArrayType` column containing a list of product tags per product. Use `F.col("tags")[0]` to get the first tag.
4. Take the employees DataFrame. Use `fillna` to replace null salaries with the average salary (compute avg first, then fill).

In [ ]:
# Exercise 1: E-commerce order schema
order_schema = StructType([
    StructField("order_id",       IntegerType(),     nullable=False),
    StructField("customer_name",  StringType(),      nullable=True),
    StructField("product_name",   StringType(),      nullable=True),
    StructField("quantity",       IntegerType(),     nullable=True),
    StructField("unit_price",     DecimalType(10,2), nullable=True),
    StructField("order_date",     DateType(),        nullable=True),
    StructField("is_returned",    BooleanType(),     nullable=True),
])

orders = [
    (1001, "Alice",  "Laptop",    1, Decimal("999.99"),  date(2024,1,15), False),
    (1002, "Bob",    "T-Shirt",   3, Decimal("29.99"),   date(2024,1,16), False),
    (1003, "Carol",  "Headphone", 1, Decimal("149.99"),  date(2024,1,17), True),
]

orders_df = spark.createDataFrame(orders, order_schema)
orders_df.printSchema()
orders_df.show(truncate=False)

In [ ]:
# Exercise 3: ArrayType column
products = [
    ("P001", "Laptop",   ["electronics", "computer", "portable"]),
    ("P002", "T-Shirt",  ["clothing", "casual", "cotton"]),
    ("P003", "Headphone",["electronics", "audio"]),
]

prod_schema = StructType([
    StructField("id",   StringType(),              nullable=False),
    StructField("name", StringType(),              nullable=True),
    StructField("tags", ArrayType(StringType()),   nullable=True),
])

prod_df = spark.createDataFrame(products, prod_schema)
prod_df.show(truncate=False)

# Access first tag
prod_df.select("name", F.col("tags")[0].alias("primary_tag"), F.size("tags").alias("tag_count")).show()